<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/hadoop_Preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openpyxl -q

import pandas as pd
import re
from google.colab import files

print("✅ Ready!")

✅ Ready!


In [ ]:
print("📁 Upload hadoop_bugs.csv")
uploaded = files.upload()

df = pd.read_csv('hadoop_bugs.csv', low_memory=False)
print(f"✅ Loaded: {df.shape}")
print("\nOriginal Priority distribution:")
print(df['Priority'].value_counts(dropna=False))

📁 Upload hadoop_bugs.csv


Saving hadoop_bugs.csv to hadoop_bugs.csv
✅ Loaded: (2503, 9)

Original Priority distribution:
Priority
Major       1738
Minor        539
Critical      86
Blocker       76
Trivial       64
Name: count, dtype: int64


In [ ]:
# Hadoop is Jira-based
# Blocker/Critical → High, Major → Medium, Minor/Trivial → Low

jira_map = {
    'Blocker':  'High',
    'Critical': 'High',
    'Major':    'Medium',
    'Minor':    'Low',
    'Trivial':  'Low'
}

df['priority'] = df['Priority'].map(jira_map)

df = df[df['priority'].notna()].copy()
df = df.reset_index(drop=True)

print(f"✅ After fixing Priority: {df.shape}")
print(df['priority'].value_counts())

✅ After fixing Priority: (2503, 10)
priority
Medium    1738
Low        603
High       162
Name: count, dtype: int64


In [ ]:
df['text'] = (
    df['Summary'].fillna('').str.strip() + ' ' +
    df['Description'].fillna('').str.strip()
)

print(f"✅ Text column created!")
print(f"Avg words (before clean): {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words (before clean): {df['text'].str.split().str.len().max()}")

✅ Text column created!
Avg words (before clean): 91
Max words (before clean): 4737


In [ ]:
def clean_text(text):
    text = str(text)

    # Remove log lines [task 2020-01-02T14:00:26Z]
    text = re.sub(
        r'\[task \d{4}-\d{2}-\d{2}T[\d:.]+Z\].*?(?=\[task|\Z)',
        '', text, flags=re.DOTALL
    )
    # Remove Jira boilerplate
    text = re.sub(
        r'(User Agent|Build Identifier|Filed by|Parsed log|Full log)\s*:.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove test log lines
    text = re.sub(
        r'(TEST-PASS|TEST-OK|TEST-FAIL|TEST-UNEXPECTED|GECKO\(\d+\)|INFO\s*-)\s*.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove Java stack traces (at com.example / at java.lang)
    text = re.sub(
        r'\s+at\s+[\w\.\$]+\([\w\.\:]+\)',
        '', text, flags=re.IGNORECASE
    )
    # Remove Hadoop-specific log patterns (WARN, ERROR, DEBUG prefixes)
    text = re.sub(
        r'\b(WARN|ERROR|DEBUG|INFO|FATAL)\b.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove memory stat lines
    text = re.sub(r'MEMORY STAT.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    # Remove file paths
    text = re.sub(
        r'[\w/\-\.]+\.(js|cpp|py|java|html|css|ts|json|xml|md|log|yaml|conf|sh)\b',
        '', text, flags=re.IGNORECASE
    )
    # Remove code blocks
    text = re.sub(r'```.*?```', ' ', text, flags=re.DOTALL)
    text = re.sub(r'`[^`\n]+`', ' ', text)
    # Remove markdown bold/italic
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*\n]+)\*', r'\1', text)
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)
    # Remove hex values
    text = re.sub(r'\b0x[0-9a-fA-F]+\b', '', text)
    # Remove timestamps
    text = re.sub(r'\b\d{2}:\d{2}:\d{2}(\.\d+)?\b', '', text)
    # Remove dates
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}\b', '', text)
    # Remove version numbers
    text = re.sub(r'\bv?\d+\.\d+(\.\d+)*\b', '', text)
    # Remove separator lines
    text = re.sub(r'[-=*_#~^|<>]{3,}', ' ', text)

    # Remove noisy lines
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        line = line.strip()
        if len(line) < 3:
            continue
        alpha_count = sum(c.isalpha() for c in line)
        if alpha_count < 3:
            continue
        if alpha_count / (len(line) + 1) < 0.3:
            continue
        clean_lines.append(line)

    text = ' '.join(clean_lines)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("⏳ Cleaning texts — please wait...")
df['text'] = df['text'].apply(clean_text)
print("✅ Text cleaning done!")
print(f"Avg words (after clean): {df['text'].str.split().str.len().mean():.0f}")

⏳ Cleaning texts — please wait...
✅ Text cleaning done!
Avg words (after clean): 76


In [ ]:
def truncate(text, max_words=100):
    return ' '.join(text.split()[:max_words])

df['text'] = df['text'].apply(truncate)

print("✅ Truncated to max 100 words!")
print(f"Avg words: {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words: {df['text'].str.split().str.len().max()}")

✅ Truncated to max 100 words!
Avg words: 53
Max words: 100


In [ ]:
df = df[df['text'].str.split().str.len() >= 5].copy()
df = df.drop_duplicates(subset='text').copy()
df = df.reset_index(drop=True)

print(f"✅ After quality filter: {df.shape}")
print(df['priority'].value_counts())

✅ After quality filter: (2434, 11)
priority
Medium    1691
Low        587
High       156
Name: count, dtype: int64


In [ ]:
label_map = {'High': 0, 'Medium': 1, 'Low': 2}
df['priority_id'] = df['priority'].map(label_map)
df['source']      = 'gitbugs_hadoop'

final = df[['text', 'priority', 'priority_id', 'source']].copy()

print("✅ Final columns set!")
print(final.head(3).to_string())

✅ Final columns set!
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      text priority  priority_id          source
0                                                                                                                                                                                JAR in conflict with timestamp ch

In [ ]:
wc = final['text'].str.split().str.len()

print("=" * 50)
print("   ✅ HADOOP_CLEAN — FINAL RESULTS")
print("=" * 50)
print(f"  Total rows     : {len(final):,}")
print(f"  Avg word count : {wc.mean():.0f}")
print(f"  Min word count : {wc.min()}")
print(f"  Max word count : {wc.max()}")
print()
print("  Priority distribution:")
print(final['priority'].value_counts().to_string())
print()
print("  Priority %:")
print((final['priority'].value_counts(normalize=True)*100).round(1).to_string())
print()
print("  Sample rows:")
for _, row in final.sample(3, random_state=42).iterrows():
    print(f"\n  [{row['priority']}] {row['text'][:120]}")
print()
print("=" * 50)

final.to_csv('hadoop_clean.csv', index=False)
files.download('hadoop_clean.csv')
print("✅ Downloaded: hadoop_clean.csv")

   ✅ HADOOP_CLEAN — FINAL RESULTS
  Total rows     : 2,434
  Avg word count : 54
  Min word count : 5
  Max word count : 100

  Priority distribution:
priority
Medium    1691
Low        587
High       156

  Priority %:
priority
Medium    69.5
Low       24.1
High       6.4

  Sample rows:

  [Medium] should not be shaded by Hadoop build is a runtime library and its references are being shaded on Hadoop, breaking the in

  [Low] Could not find artifact org.apache.ftpserve:ftplet-api:jar: Hi, i have a problem with fetch POM file for ftplet-api for 

  [Medium] Javadoc errors are always ignored in the precommit jobs There are many javadoc errors reported and fixed. This is mostly



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: hadoop_clean.csv
